In [ ]:
import os
from accelerate.utils import write_basic_config

In [ ]:
!huggingface-cli whoami

In [ ]:
!nvidia-smi

In [ ]:
from datasets import load_dataset
dataset = load_dataset("/EuroSAT", split='train')

In [ ]:
dataset.features

In [ ]:
len(dataset)

In [ ]:
class_names = ['Annual Crop', 'Forest', 'Herbaceous Vegetation', 'Highway', 'Industrial Buildings', 'Pasture', 'Permanent Crop', 'Residential Buildings', 'River', 'SeaLake']
class_dict = {index: name for index, name in enumerate(class_names)}

class_dict

In [ ]:
new_dataset = dataset
caption_column = ["caption"] * len(new_dataset)
new_dataset = new_dataset.add_column("caption", caption_column)
new_dataset.features

In [ ]:
new_dataset[0]['image']

In [ ]:
new_dataset[0]['label']

In [ ]:
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
import torch
from PIL import Image
import requests

model = InstructBlipForConditionalGeneration.from_pretrained("Mediocreatmybest/instructblip-vicuna-7b_8bit")
processor = InstructBlipProcessor.from_pretrained("Mediocreatmybest/instructblip-vicuna-7b_8bit")
device = "cuda" if torch.cuda.is_available() else "cpu"
#model.to(device)

In [ ]:
def meta_prompt(label):
    prompt = f"Generate a concise caption for the provided remote sensing image, focusing on the specified {label} class. In your caption, accurately highlight the distinct features observed in the image."
    prompt = "Generate a concise caption for the provided remote sensing image that encapsulates the essence of the ecosystem depicted. Focus on the type of ecosystem and its intrinsic characteristics, such as the Vegetation, Terrain, and Climate conditions. Describe the interplay of Biodiversity Components and the services they provide, reflected as Ecosystem Services. If there are indications of Disturbances—whether from natural events like Climate Events or human activities—detail their impact on the ecosystem's Ecosystem Dynamics, including any visible signs of resilience or vulnerability."
    return prompt

In [ ]:
image = new_dataset[0]['image']
label = class_dict[new_dataset[0]['label']]
prompt = meta_prompt(label)
prompt

In [ ]:
image = new_dataset[100]['image']
label = class_dict[new_dataset[100]['label']]
prompt = meta_prompt(label)
inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    do_sample=True,
    num_beams=5,
    max_length=256,
    min_length=10,
    top_p=0.8,
    repetition_penalty=1.5,
    length_penalty=1.0,
    temperature=0.5,
)
generated_text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()
print(generated_text)

In [ ]:
def caption_image(image, label):
    label = class_dict[label]
    prompt = meta_prompt(label)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        do_sample=True,
        num_beams=5,
        max_length=256,
        min_length=10,
        top_p=0.9,
        repetition_penalty=1.5,
        length_penalty=1.0,
        temperature=0.9,
    )
    return processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()

In [ ]:
def modify_caption(item):
    image_to_caption = item['image']
    label_to_use = item['label']
    caption = caption_image(image_to_caption, label_to_use)
    item['caption'] = caption
    return item

In [ ]:
test = caption_image(new_dataset[10]['image'], new_dataset[10]['label'])
test

In [ ]:
modified_dataset = new_dataset.map(modify_caption)

In [ ]:
modified_dataset[301]['label']

In [ ]:
modified_dataset

In [ ]:
modified_dataset.push_to_hub("/EuroSAT-InstructCaptions", private=False)

In [ ]:
# BLIP - OLD

In [ ]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large", torch_dtype=torch.float16).to("cuda")

In [ ]:
def caption_image(image):
    text = "a satellite image"
    inputs = processor(image, text, return_tensors="pt").to("cuda")

    out = model.generate(**inputs)
    return processor.decode(out[0], skip_special_tokens=True)

In [ ]:
test = caption_image(new_dataset[10]['image'])
test

In [ ]:
def modify_caption(item):
    image_to_caption = item['image']
    caption = caption_image(image_to_caption)
    item['caption'] = caption
    return item

In [ ]:
modified_dataset = new_dataset.map(modify_caption)

In [ ]:
modified_dataset[0]['caption']

In [ ]:
modified_dataset.push_to_hub("<redacted>/EuroSAT-captions", private=True)